# Part 3: Scalable Processing — Polars Query Optimization, Pandas Benchmark & PySpark Parity
Examines Polars lazy query optimization (predicate pushdown, projection pushdown, parallel window execution), benchmarks execution time and peak RSS against Pandas, and verifies exact parity with PySpark.

In [1]:
from pathlib import Path
print(Path('../outputs/tables/polars_optimized_plan.txt').read_text())

=== UNOPTIMIZED LOGICAL PLAN ===
 WITH_COLUMNS:
 [when(col("qidx").shift([dyn int: 4]).over([col("CERT")]) == (col("qidx") - 4)).then((col("ASSET") / col("ASSET").shift([dyn int: 4]).over([col("CERT")])) - 1.0).otherwise(null.cast(Float64)).alias("asset_growth_4q"), when(col("qidx").shift([dyn int: 4]).over([col("CERT")]) == (col("qidx") - 4)).then(col("texas_ratio") - col("texas_ratio").shift([dyn int: 4]).over([col("CERT")])).otherwise(null.cast(Float64)).alias("d4_texas_ratio")] 
   WITH_COLUMNS:
   [when(col("quarter") == 1).then(col("NETINC")).otherwise(when(col("qidx").shift([dyn int: 1]).over([col("CERT"), col("year")]) == (col("qidx") - 1)).then(col("NETINC") - col("NETINC").shift([dyn int: 1]).over([col("CERT"), col("year")])).otherwise(null.cast(Float64))).alias("NETINC_Q"), (col("EQ") / col("ASSET")).alias("eq_asset_ratio"), when((col("EQ").fill_null([0.0]) + col("LNATRES").fill_null([0.0])) > 0.0).then((((col("NCLNLS").fill_null([0.0]) + col("ORE").fill_null([0.0])) / (col(

In [2]:
import polars as pl
print(pl.read_csv('../outputs/tables/part3_benchmark_results.csv'))

slice,rows,engine,time_s,peak_rss_mb,max_rel_diff_vs_polars_lazy
2010-2026,386056,Polars lazy,0.301,371.0,0.0
2010-2026,386056,Polars streaming,0.138,354.0,2.3e-16
2010-2026,386056,Polars eager,0.343,547.0,0.0
2010-2026,386056,pandas + pushdown,0.287,519.0,2.3e-16
2010-2026,386056,pandas naive,0.861,1653.0,2.3e-16
1995-2026,977097,Polars lazy,0.717,544.0,0.0
1995-2026,977097,Polars streaming,0.332,1102.0,2.3e-16
1995-2026,977097,Polars eager,0.728,672.0,0.0
1995-2026,977097,pandas + pushdown,0.424,743.0,1.3e-16
1995-2026,977097,pandas naive,1.302,1822.0,1.3e-16
1984-2026 (full),1682616,Polars lazy,1.264,686.0,0.0
1984-2026 (full),1682616,Polars streaming,0.554,1232.0,3.2e-16
1984-2026 (full),1682616,Polars eager,1.205,786.0,0.0
1984-2026 (full),1682616,pandas + pushdown,0.607,1038.0,2.4e-16
1984-2026 (full),1682616,pandas naive,1.187,1938.0,2.4e-16



In [3]:
print('--- PySpark vs Polars Parity Check ---')
print(Path('../outputs/tables/spark_polars_parity.json').read_text())
print('--- PySpark Catalyst Execution Plan ---')
print(Path('../outputs/tables/spark_execution_plan.txt').read_text()[:1500])

--- PySpark vs Polars Parity Check ---
{
  "yearly_fingerprint": {
    "years_compared": 43,
    "row_and_null_counts_identical": true,
    "max_relative_diff_of_sums": 2.2382624412157772e-15
  },
  "row_level_sample": {
    "banks_sampled": 483,
    "rows_compared": 33042,
    "rows_only_in_one_engine": 0,
    "max_abs_diff_any_column": 0.0
  },
  "timing_seconds": {
    "polars_lazy_collect_plus_aggregate": 0.6,
    "spark_session_startup": 3.9,
    "spark_job_cold": 4.84,
    "spark_job_warm": 1.93,
    "note": "single in-process runs on the full panel; Spark = local[2], 1.2 GB driver"
  }
}
--- PySpark Catalyst Execution Plan ---
== Physical Plan ==
AdaptiveSparkPlan (12)
+- Project (11)
   +- Window (10)
      +- Sort (9)
         +- Exchange (8)
            +- Project (7)
               +- Window (6)
                  +- Sort (5)
                     +- Exchange (4)
                        +- Project (3)
                           +- Project (2)
                              +- S